# Interactive Fiducial Drift Correction

This notebook demonstrates interactive fiducial marker selection and drift correction using the new `DriftCorrectionFunctions.py` module.

**Fiducial-based drift correction is ideal for:**
- Experiments with gold nanoparticles, fluorescent beads, or other stationary markers
- High precision drift correction requirements  
- Data where correlation-based methods fail due to sparse labeling
- Multi-color experiments requiring precise registration

**Features:**
- Interactive plotting for fiducial visualization and selection
- Automatic fiducial detection with parameter tuning
- Manual fiducial picking with visual confirmation
- Quality assessment and comparison tools
- Comprehensive result export and analysis

**Author:** Claude Code Assistant  
**Created:** September 3, 2025

## Setup and Imports

In [ ]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
import warnings
from typing import Tuple, Dict, Any, List, Optional
from dataclasses import dataclass
import json

# Configure matplotlib for notebook
%matplotlib widget
plt.style.use("default")

# Add src to path
sys.path.append("../../src")

# Import pyBayerSMLM modules
import DriftCorrectionFunctions as DCF
import IOFunctions
import render

print("✅ All imports successful")
print(
    f"Available drift correction methods: {DCF.Drift_Correction_Functions().available_methods()}"
)

## Data Structures and Helper Functions

In [ ]:
@dataclass
class FiducialSelectionResult:
    """Results from interactive fiducial selection."""

    picked_coordinates: List[Tuple[float, float]]
    selected_localizations: np.recarray
    n_fiducials: int
    selection_method: str
    parameters_used: Dict[str, Any]


def create_example_data() -> Tuple[np.recarray, List[dict]]:
    """
    Create example data with simulated fiducials for demonstration.

    Returns:
        Tuple of (localizations, metadata_info)
    """
    print("Creating example data with simulated fiducials...")

    np.random.seed(42)

    # Simulation parameters
    n_frames = 1000
    image_size = 128  # 128x128 pixel image
    pixel_size = 0.1  # 100nm pixels

    # Create 4 fiducial markers at known positions
    fiducial_positions = [
        (20, 20),  # Bottom-left
        (108, 20),  # Bottom-right
        (20, 108),  # Top-left
        (108, 108),  # Top-right
    ]

    # Simulate drift
    frames = np.arange(n_frames)
    true_drift_x = 0.8 * np.sin(frames / 100) + 0.003 * frames
    true_drift_y = 0.6 * np.cos(frames / 80) + 0.002 * frames

    all_locs = []

    # Generate fiducial localizations (high density, stable)
    for fid_id, (base_x, base_y) in enumerate(fiducial_positions):
        n_locs_per_frame = 25  # Dense fiducials

        for frame in range(n_frames):
            if np.random.random() > 0.05:  # 95% probability of detection per frame
                n_this_frame = np.random.poisson(n_locs_per_frame)

                # Base positions with small random scatter
                x_coords = base_x + np.random.normal(0, 0.4, n_this_frame)
                y_coords = base_y + np.random.normal(0, 0.4, n_this_frame)

                # Add drift
                x_coords += true_drift_x[frame]
                y_coords += true_drift_y[frame]

                # Add localization precision noise
                x_coords += np.random.normal(0, 0.02, n_this_frame)
                y_coords += np.random.normal(0, 0.02, n_this_frame)

                # Create localization records
                for x, y in zip(x_coords, y_coords):
                    all_locs.append(
                        {
                            "xc": x * pixel_size,  # Convert to micrometers
                            "yc": y * pixel_size,
                            "frame": frame,
                            "photons": np.random.exponential(2500),
                            "is_fiducial": True,
                            "fiducial_id": fid_id,
                        }
                    )

    # Generate background cellular structures
    n_bg_structures = 8

    for struct_id in range(n_bg_structures):
        # Random positions away from fiducials
        while True:
            base_x = np.random.uniform(35, 93)
            base_y = np.random.uniform(35, 93)

            # Check distance from fiducials
            min_dist = min(
                [
                    np.sqrt((base_x - fx) ** 2 + (base_y - fy) ** 2)
                    for fx, fy in fiducial_positions
                ]
            )
            if min_dist > 12:  # At least 12 pixels from fiducials
                break

        for frame in range(n_frames):
            if np.random.random() > 0.6:  # 40% probability per frame
                n_this_frame = np.random.poisson(8)  # Moderate background

                # Larger scatter for biological structures
                x_coords = base_x + np.random.normal(0, 4, n_this_frame)
                y_coords = base_y + np.random.normal(0, 4, n_this_frame)

                # Add drift
                x_coords += true_drift_x[frame]
                y_coords += true_drift_y[frame]

                # Add precision noise
                x_coords += np.random.normal(0, 0.06, n_this_frame)
                y_coords += np.random.normal(0, 0.06, n_this_frame)

                for x, y in zip(x_coords, y_coords):
                    all_locs.append(
                        {
                            "xc": x * pixel_size,
                            "yc": y * pixel_size,
                            "frame": frame,
                            "photons": np.random.exponential(1200),
                            "is_fiducial": False,
                            "fiducial_id": -1,
                        }
                    )

    # Convert to structured array
    locs_df = pd.DataFrame(all_locs)
    locs = np.rec.fromrecords(
        locs_df[["xc", "yc", "frame", "photons"]].values,
        names=["xc", "yc", "frame", "photons"],
    )

    # Create metadata
    info = [
        {
            "Width": image_size,
            "Height": image_size,
            "Frames": n_frames,
            "Pixelsize": pixel_size,
        }
    ]

    print(f"✅ Created example data:")
    print(f"   - {len(locs):,} total localizations")
    print(f"   - {len(fiducial_positions)} simulated fiducials")
    print(f"   - {n_frames} frames")
    print(
        f"   - Image size: {image_size}x{image_size} pixels ({pixel_size*1000:.0f} nm/pixel)"
    )
    print(
        f"   - Simulated drift range: X=[{true_drift_x.min():.3f}, {true_drift_x.max():.3f}], Y=[{true_drift_y.min():.3f}, {true_drift_y.max():.3f}] pixels"
    )

    return locs, info


print("✅ Helper functions defined")

## Load or Create Data

**Option 1:** Use the example data generator below  
**Option 2:** Load your own data by modifying the cell below

In [ ]:
# Option 1: Create example data with simulated fiducials
locs, info = create_example_data()

# Option 2: Load your own data (uncomment and modify as needed)
# io = IOFunctions.IO_Functions()
# locs = io.read_localisations("your_localizations.csv")
# info = [{"Width": 256, "Height": 256, "Frames": 10000, "Pixelsize": 0.1}]

print(f"Data loaded: {len(locs):,} localizations")

## Visualize Localizations

Create an overview image to see the structure and identify potential fiducials.

In [ ]:
def create_localization_overview(locs, info):
    """Create an overview image from localizations."""

    # Extract metadata
    meta = DCF.CoordinateProcessor.extract_metadata(info)
    width = meta["width"]
    height = meta["height"]
    pixelsize = meta.get("pixelsize", 0.1)

    # Try to use render module
    try:
        image = render.render(
            locs=locs, info=info, blur_method="gaussian", min_blur_width=1.0
        )[
            1
        ]  # Take the rendered image

        print(f"✅ Image rendered using render module: {image.shape}")

    except Exception as e:
        print(f"⚠️ Render module failed ({e}), creating histogram image...")

        # Fallback: create 2D histogram
        x_pixels = np.clip((locs.xc / pixelsize).astype(int), 0, width - 1)
        y_pixels = np.clip((locs.yc / pixelsize).astype(int), 0, height - 1)

        image = np.zeros((height, width), dtype=np.float32)

        # Add localizations with photon weighting
        if hasattr(locs, "photons"):
            weights = locs.photons
        else:
            weights = np.ones(len(locs))

        for x, y, w in zip(x_pixels, y_pixels, weights):
            image[y, x] += w

    # Display the image
    fig, ax = plt.subplots(figsize=(10, 10))

    extent = [0, width * pixelsize, 0, height * pixelsize]
    im = ax.imshow(
        image, extent=extent, origin="lower", cmap="hot", interpolation="nearest"
    )

    ax.set_xlabel("X (μm)")
    ax.set_ylabel("Y (μm)")
    ax.set_title(f"Localization Overview\n{len(locs):,} localizations")

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label("Intensity")

    plt.tight_layout()
    plt.show()

    return image


# Create overview
overview_image = create_localization_overview(locs, info)

## Method 1: Automatic Fiducial Detection

Use the built-in automatic detection algorithm to find fiducial markers. This is the recommended approach for most datasets.

In [ ]:
def automatic_fiducial_detection(
    locs,
    info,
    threshold_percentile=92.0,
    box_size_nm=1000.0,
    min_frames_fraction=0.6,
    histogram_bins=256,
):
    """Automatically detect fiducial markers."""

    print("\n" + "=" * 60)
    print("AUTOMATIC FIDUCIAL DETECTION")
    print("=" * 60)

    print(f"Detection parameters:")
    print(f"  - Threshold percentile: {threshold_percentile}%")
    print(f"  - Box size: {box_size_nm} nm")
    print(f"  - Min frames fraction: {min_frames_fraction}")
    print(f"  - Histogram bins: {histogram_bins}")

    # Use the built-in detection method
    drift_corrector = DCF.Drift_Correction_Functions()

    try:
        corrected_locs, drift_result, detection_info = (
            drift_corrector.undrift_with_fiducial_detection(
                locs=locs,
                info=info,
                threshold_percentile=threshold_percentile,
                box_size_nm=box_size_nm,
                min_frames_fraction=min_frames_fraction,
                histogram_bins=histogram_bins,
            )
        )

        print(f"✅ Automatic detection completed")
        print(f"   Found {detection_info['n_fiducials']} fiducials")
        print(f"   Frames per fiducial: {detection_info['frames_per_fiducial']}")

        # Extract fiducial coordinates
        picked_coords = []
        if hasattr(corrected_locs, "group"):
            unique_groups = np.unique(corrected_locs.group)

            for group_id in unique_groups:
                if group_id >= 0:  # Valid fiducial groups
                    group_locs = corrected_locs[corrected_locs.group == group_id]
                    if len(group_locs) > 0:
                        # Calculate center of mass
                        center_x = np.mean(group_locs.xc)
                        center_y = np.mean(group_locs.yc)
                        picked_coords.append((center_x, center_y))

        result = FiducialSelectionResult(
            picked_coordinates=picked_coords,
            selected_localizations=corrected_locs,
            n_fiducials=detection_info["n_fiducials"],
            selection_method="automatic",
            parameters_used={
                "threshold_percentile": threshold_percentile,
                "box_size_nm": box_size_nm,
                "min_frames_fraction": min_frames_fraction,
                "histogram_bins": histogram_bins,
            },
        )

        return result, drift_result, detection_info

    except Exception as e:
        print(f"❌ Automatic detection failed: {e}")
        print("\nTroubleshooting tips:")
        print("  - Try lowering threshold_percentile (e.g., 85-90%)")
        print("  - Try reducing min_frames_fraction (e.g., 0.4-0.5)")
        print("  - Try increasing box_size_nm (e.g., 1200-1500nm)")
        raise


# Run automatic detection with adjustable parameters
print("Running automatic fiducial detection...")
print("(You can adjust the parameters below if detection fails)")

# You can adjust these parameters if needed:
auto_result, auto_drift, auto_info = automatic_fiducial_detection(
    locs,
    info,
    threshold_percentile=90.0,  # Lower = more candidates
    box_size_nm=1200.0,  # Larger = wider detection area
    min_frames_fraction=0.5,  # Lower = accept fiducials with fewer localizations
    histogram_bins=256,
)

print(f"\n✅ Automatic detection successful!")
print(f"Found {auto_result.n_fiducials} fiducials at positions:")
for i, (x, y) in enumerate(auto_result.picked_coordinates):
    print(f"  Fiducial {i+1}: ({x:.2f}, {y:.2f}) μm")

## Preview Detected Fiducials

Visualize the automatically detected fiducials on the localization image.

In [ ]:
def preview_fiducials(locs, info, selection_result):
    """Show preview of selected fiducials."""

    print(f"Previewing {selection_result.n_fiducials} selected fiducials...")

    # Extract metadata
    meta = DCF.CoordinateProcessor.extract_metadata(info)
    width = meta["width"]
    height = meta["height"]
    pixelsize = meta.get("pixelsize", 0.1)

    # Create or get overview image
    try:
        image = render.render(
            locs=locs, info=info, blur_method="gaussian", min_blur_width=1.0
        )[1]
    except:
        # Fallback histogram method
        x_pixels = np.clip((locs.xc / pixelsize).astype(int), 0, width - 1)
        y_pixels = np.clip((locs.yc / pixelsize).astype(int), 0, height - 1)
        image = np.zeros((height, width), dtype=np.float32)
        for x, y in zip(x_pixels, y_pixels):
            image[y, x] += 1

    # Create preview plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

    # Left: Overview with fiducial markers
    extent = [0, width * pixelsize, 0, height * pixelsize]
    ax1.imshow(image, extent=extent, origin="lower", cmap="hot", alpha=0.8)

    # Overlay fiducial selections
    for i, (x, y) in enumerate(selection_result.picked_coordinates):
        circle = Circle((x, y), 0.6, fill=False, color="cyan", linewidth=3)
        ax1.add_patch(circle)
        ax1.text(
            x,
            y + 0.8,
            f"F{i+1}",
            color="cyan",
            ha="center",
            va="bottom",
            fontweight="bold",
            fontsize=12,
        )

    ax1.set_xlabel("X (μm)")
    ax1.set_ylabel("Y (μm)")
    ax1.set_title(
        f"Detected Fiducials ({selection_result.selection_method})\n"
        f"{selection_result.n_fiducials} fiducials found"
    )

    # Right: Fiducial localizations only (colored by group)
    if hasattr(selection_result.selected_localizations, "group"):
        fiducial_locs = selection_result.selected_localizations[
            selection_result.selected_localizations.group >= 0
        ]

        if len(fiducial_locs) > 0:
            unique_groups = np.unique(fiducial_locs.group)
            colors = plt.cm.Set1(np.linspace(0, 1, len(unique_groups)))

            for i, group_id in enumerate(unique_groups):
                group_locs = fiducial_locs[fiducial_locs.group == group_id]
                ax2.scatter(
                    group_locs.xc,
                    group_locs.yc,
                    s=3,
                    alpha=0.7,
                    c=[colors[i]],
                    label=f"Fiducial {group_id+1}",
                    rasterized=True,
                )

    ax2.set_xlabel("X (μm)")
    ax2.set_ylabel("Y (μm)")
    ax2.set_title("Fiducial Localizations Only")
    ax2.legend()
    ax2.set_aspect("equal")

    plt.tight_layout()
    plt.show()


# Preview the automatically detected fiducials
preview_fiducials(locs, info, auto_result)

## Analyze Detection Quality

Examine the quality and characteristics of the detected fiducials.

In [ ]:
def analyze_fiducial_quality(selection_result, drift_result):
    """Analyze the quality of fiducial detection and drift correction."""

    print("\n" + "=" * 60)
    print("FIDUCIAL QUALITY ANALYSIS")
    print("=" * 60)

    # Extract fiducial localizations
    locs_with_groups = selection_result.selected_localizations
    fiducial_locs = locs_with_groups[locs_with_groups.group >= 0]

    if len(fiducial_locs) == 0:
        print("❌ No fiducial localizations found for analysis")
        return {}

    # Calculate per-fiducial statistics
    unique_groups = np.unique(fiducial_locs.group)
    fiducial_stats = []

    print(f"Per-Fiducial Statistics:")
    print(
        f"{'ID':<3} {'Locs':<6} {'Center (X,Y)':<15} {'Precision (nm)':<15} {'Frames':<7}"
    )
    print("-" * 55)

    for group_id in unique_groups:
        group_locs = fiducial_locs[fiducial_locs.group == group_id]

        # Calculate statistics
        center_x = np.mean(group_locs.xc)
        center_y = np.mean(group_locs.yc)
        std_x = np.std(group_locs.xc)
        std_y = np.std(group_locs.yc)
        frames_present = len(np.unique(group_locs.frame))

        fiducial_stats.append(
            {
                "group_id": group_id,
                "n_localizations": len(group_locs),
                "center_x": center_x,
                "center_y": center_y,
                "std_x": std_x,
                "std_y": std_y,
                "frames_present": frames_present,
            }
        )

        print(
            f"{group_id+1:<3} {len(group_locs):<6} ({center_x:.2f},{center_y:.2f}) "
            f"({std_x*1000:.0f},{std_y*1000:.0f})     {frames_present:<7}"
        )

    # Overall quality metrics
    all_std_x = [f["std_x"] for f in fiducial_stats]
    all_std_y = [f["std_y"] for f in fiducial_stats]

    metrics = {
        "n_fiducials": len(fiducial_stats),
        "total_fiducial_localizations": len(fiducial_locs),
        "mean_localizations_per_fiducial": np.mean(
            [f["n_localizations"] for f in fiducial_stats]
        ),
        "mean_precision_x_nm": np.mean(all_std_x) * 1000,
        "mean_precision_y_nm": np.mean(all_std_y) * 1000,
        "max_drift_magnitude": np.sqrt(
            drift_result.drift_x**2 + drift_result.drift_y**2
        ).max(),
        "drift_x_range": drift_result.drift_x.max() - drift_result.drift_x.min(),
        "drift_y_range": drift_result.drift_y.max() - drift_result.drift_y.min(),
        "fiducial_stats": fiducial_stats,
        "selection_method": selection_result.selection_method,
    }

    print(f"\nOverall Quality Summary:")
    print(f"  - Number of fiducials: {metrics['n_fiducials']}")
    print(
        f"  - Total fiducial localizations: {metrics['total_fiducial_localizations']:,}"
    )
    print(
        f"  - Average localizations per fiducial: {metrics['mean_localizations_per_fiducial']:.0f}"
    )
    print(
        f"  - Mean fiducial precision: {metrics['mean_precision_x_nm']:.0f} nm (X), {metrics['mean_precision_y_nm']:.0f} nm (Y)"
    )

    print(f"\nDrift Correction Results:")
    print(f"  - Maximum drift magnitude: {metrics['max_drift_magnitude']:.3f} pixels")
    print(f"  - X drift range: {metrics['drift_x_range']:.3f} pixels")
    print(f"  - Y drift range: {metrics['drift_y_range']:.3f} pixels")

    return metrics


# Analyze the fiducial quality
quality_metrics = analyze_fiducial_quality(auto_result, auto_drift)

## Visualize Drift Correction Results

Create comprehensive plots showing the before/after comparison and drift traces.

In [ ]:
def plot_drift_correction_results(
    original_locs, corrected_locs, selection_result, drift_result
):
    """Create comprehensive drift correction result plots."""

    fig = plt.figure(figsize=(18, 12))
    gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

    # 1. Original data with fiducial markers
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.scatter(
        original_locs.xc, original_locs.yc, s=0.5, alpha=0.3, c="red", rasterized=True
    )

    # Highlight fiducial regions
    for i, (x, y) in enumerate(selection_result.picked_coordinates):
        circle = Circle((x, y), 0.5, fill=False, color="cyan", linewidth=2)
        ax1.add_patch(circle)
        ax1.text(
            x,
            y + 0.6,
            f"F{i+1}",
            color="cyan",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

    ax1.set_title("Original Data + Fiducials")
    ax1.set_xlabel("X (μm)")
    ax1.set_ylabel("Y (μm)")
    ax1.set_aspect("equal")

    # 2. Corrected data
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.scatter(
        corrected_locs.xc,
        corrected_locs.yc,
        s=0.5,
        alpha=0.3,
        c="blue",
        rasterized=True,
    )
    ax2.set_title("After Fiducial Correction")
    ax2.set_xlabel("X (μm)")
    ax2.set_ylabel("Y (μm)")
    ax2.set_aspect("equal")

    # 3. Drift traces
    ax3 = fig.add_subplot(gs[0, 2])
    frames = np.arange(len(drift_result.drift_x))
    ax3.plot(
        frames, drift_result.drift_x, "b-", linewidth=1, label="X drift", alpha=0.8
    )
    ax3.plot(
        frames, drift_result.drift_y, "r-", linewidth=1, label="Y drift", alpha=0.8
    )
    ax3.set_title("Measured Drift")
    ax3.set_xlabel("Frame")
    ax3.set_ylabel("Drift (pixels)")
    ax3.legend()
    ax3.grid(True, alpha=0.3)

    # 4. Fiducial localizations before correction (colored by group)
    ax4 = fig.add_subplot(gs[1, 0])
    if hasattr(selection_result.selected_localizations, "group"):
        fiducial_locs = selection_result.selected_localizations[
            selection_result.selected_localizations.group >= 0
        ]

        if len(fiducial_locs) > 0:
            unique_groups = np.unique(fiducial_locs.group)
            colors = plt.cm.Set1(np.linspace(0, 1, len(unique_groups)))

            for i, group_id in enumerate(unique_groups):
                group_locs = fiducial_locs[fiducial_locs.group == group_id]
                ax4.scatter(
                    group_locs.xc,
                    group_locs.yc,
                    s=2,
                    alpha=0.6,
                    c=[colors[i]],
                    label=f"F{group_id+1}",
                    rasterized=True,
                )

    ax4.set_title("Fiducials Before Correction")
    ax4.set_xlabel("X (μm)")
    ax4.set_ylabel("Y (μm)")
    ax4.legend()
    ax4.set_aspect("equal")

    # 5. Fiducial localizations after correction
    ax5 = fig.add_subplot(gs[1, 1])
    corrected_fiducial_locs = (
        corrected_locs[corrected_locs.group >= 0]
        if hasattr(corrected_locs, "group")
        else []
    )

    if len(corrected_fiducial_locs) > 0:
        unique_groups = np.unique(corrected_fiducial_locs.group)
        for i, group_id in enumerate(unique_groups):
            group_locs = corrected_fiducial_locs[
                corrected_fiducial_locs.group == group_id
            ]
            ax5.scatter(
                group_locs.xc,
                group_locs.yc,
                s=2,
                alpha=0.6,
                c=[colors[i]],
                label=f"F{group_id+1}",
                rasterized=True,
            )

    ax5.set_title("Fiducials After Correction")
    ax5.set_xlabel("X (μm)")
    ax5.set_ylabel("Y (μm)")
    ax5.legend()
    ax5.set_aspect("equal")

    # 6. Drift magnitude over time
    ax6 = fig.add_subplot(gs[1, 2])
    drift_magnitude = np.sqrt(drift_result.drift_x**2 + drift_result.drift_y**2)
    ax6.plot(frames, drift_magnitude, "g-", linewidth=1, alpha=0.8)
    ax6.set_title("Drift Magnitude")
    ax6.set_xlabel("Frame")
    ax6.set_ylabel("Drift Magnitude (pixels)")
    ax6.grid(True, alpha=0.3)

    # Add statistics text
    stats_text = f"Max drift: {drift_magnitude.max():.3f} px\nFiducials: {selection_result.n_fiducials}\nMethod: {selection_result.selection_method}"
    ax6.text(
        0.02,
        0.98,
        stats_text,
        transform=ax6.transAxes,
        verticalalignment="top",
        fontsize=9,
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8),
    )

    # Overall title
    fig.suptitle("Fiducial Drift Correction Results", fontsize=16, fontweight="bold")

    plt.show()


# Plot the comprehensive results
plot_drift_correction_results(
    locs, auto_result.selected_localizations, auto_result, auto_drift
)

## Save Results

Export the drift correction results for further analysis or publication.

In [ ]:
def save_drift_correction_results(
    selection_result,
    corrected_locs,
    drift_result,
    quality_metrics,
    output_base="fiducial_drift_correction_results",
):
    """Save all drift correction results to files."""

    print("\n" + "=" * 60)
    print("SAVING DRIFT CORRECTION RESULTS")
    print("=" * 60)

    file_paths = []

    # 1. Save corrected localizations
    corrected_df = pd.DataFrame(corrected_locs)
    corrected_path = f"{output_base}_corrected_localizations.csv"
    corrected_df.to_csv(corrected_path, index=False)
    file_paths.append(corrected_path)
    print(f"✅ Corrected localizations: {corrected_path}")

    # 2. Save drift trace
    drift_df = pd.DataFrame(
        {
            "frame": np.arange(len(drift_result.drift_x)),
            "drift_x_pixels": drift_result.drift_x,
            "drift_y_pixels": drift_result.drift_y,
            "drift_magnitude": np.sqrt(
                drift_result.drift_x**2 + drift_result.drift_y**2
            ),
        }
    )
    drift_path = f"{output_base}_drift_trace.csv"
    drift_df.to_csv(drift_path, index=False)
    file_paths.append(drift_path)
    print(f"✅ Drift trace: {drift_path}")

    # 3. Save fiducial selection information
    selection_info = {
        "selection_method": selection_result.selection_method,
        "n_fiducials": selection_result.n_fiducials,
        "parameters_used": selection_result.parameters_used,
        "fiducial_coordinates": selection_result.picked_coordinates,
        "drift_method": drift_result.method,
        "correction_timestamp": pd.Timestamp.now().isoformat(),
    }

    selection_path = f"{output_base}_selection_info.json"
    with open(selection_path, "w") as f:
        json.dump(selection_info, f, indent=2, default=str)
    file_paths.append(selection_path)
    print(f"✅ Selection info: {selection_path}")

    # 4. Save quality metrics
    quality_path = f"{output_base}_quality_metrics.json"
    with open(quality_path, "w") as f:
        json.dump(quality_metrics, f, indent=2, default=str)
    file_paths.append(quality_path)
    print(f"✅ Quality metrics: {quality_path}")

    # 5. Create summary report
    summary_path = f"{output_base}_summary_report.txt"
    with open(summary_path, "w") as f:
        f.write("FIDUCIAL DRIFT CORRECTION SUMMARY REPORT\n")
        f.write("=" * 50 + "\n\n")

        f.write(f"Analysis Date: {pd.Timestamp.now()}\n")
        f.write(f"Selection Method: {selection_result.selection_method}\n")
        f.write(f"Number of Fiducials: {selection_result.n_fiducials}\n\n")

        f.write("DRIFT CORRECTION RESULTS:\n")
        f.write(
            f"  - Maximum drift magnitude: {quality_metrics.get('max_drift_magnitude', 0):.3f} pixels\n"
        )
        f.write(
            f"  - X drift range: {quality_metrics.get('drift_x_range', 0):.3f} pixels\n"
        )
        f.write(
            f"  - Y drift range: {quality_metrics.get('drift_y_range', 0):.3f} pixels\n\n"
        )

        f.write("FIDUCIAL QUALITY:\n")
        f.write(
            f"  - Total fiducial localizations: {quality_metrics.get('total_fiducial_localizations', 0):,}\n"
        )
        f.write(
            f"  - Mean localizations per fiducial: {quality_metrics.get('mean_localizations_per_fiducial', 0):.0f}\n"
        )
        f.write(
            f"  - Mean precision: {quality_metrics.get('mean_precision_x_nm', 0):.0f} nm (X), {quality_metrics.get('mean_precision_y_nm', 0):.0f} nm (Y)\n\n"
        )

        f.write("PARAMETERS USED:\n")
        for key, value in selection_result.parameters_used.items():
            f.write(f"  - {key}: {value}\n")

        f.write("\nFILES GENERATED:\n")
        for path in file_paths:
            f.write(f"  - {path}\n")

    file_paths.append(summary_path)
    print(f"✅ Summary report: {summary_path}")

    print(f"\n✅ All results saved successfully!")
    print(f"Generated {len(file_paths)} files with base name: {output_base}")

    return file_paths


# Save all results
saved_files = save_drift_correction_results(
    auto_result,
    auto_result.selected_localizations,  # This includes the corrected localizations
    auto_drift,
    quality_metrics,
    output_base="notebook_fiducial_drift_results",
)

print(f"\nFiles saved: {len(saved_files)} total")

## Method 2: Manual Fiducial Selection (Optional)

If automatic detection doesn't work well for your data, you can manually select fiducials by clicking on the image.

**Note:** This requires an interactive matplotlib backend. If you get errors, use the automatic detection method above.

In [ ]:
class ManualFiducialSelector:
    """Interactive tool for manual fiducial selection."""

    def __init__(self, locs, info):
        self.locs = locs
        self.info = info
        self.picked_coordinates = []
        self.fig = None
        self.ax = None
        self.circles = []

        # Extract metadata
        meta = DCF.CoordinateProcessor.extract_metadata(info)
        self.width = meta["width"]
        self.height = meta["height"]
        self.pixelsize = meta.get("pixelsize", 0.1)

    def start_selection(self, pick_radius_nm=600.0, min_localizations=40):
        """Start interactive fiducial selection."""

        print(f"\nStarting manual fiducial selection...")
        print(f"Parameters: radius={pick_radius_nm} nm, min_locs={min_localizations}")
        print(f"\nInstructions:")
        print(f"  - Click on fiducial markers in the plot")
        print(f"  - Press 'r' to remove the last selection")
        print(f"  - Press 'q' or close the window when finished")

        # Create overview image
        try:
            image = render.render(
                locs=self.locs, info=self.info, blur_method="gaussian"
            )[1]
        except:
            # Fallback method
            x_pixels = np.clip(
                (self.locs.xc / self.pixelsize).astype(int), 0, self.width - 1
            )
            y_pixels = np.clip(
                (self.locs.yc / self.pixelsize).astype(int), 0, self.height - 1
            )
            image = np.zeros((self.height, self.width), dtype=np.float32)
            for x, y in zip(x_pixels, y_pixels):
                image[y, x] += 1

        # Create interactive plot
        self.fig, self.ax = plt.subplots(figsize=(12, 12))

        extent = [0, self.width * self.pixelsize, 0, self.height * self.pixelsize]
        self.ax.imshow(
            image, extent=extent, origin="lower", cmap="hot", interpolation="nearest"
        )
        self.ax.set_xlabel("X (μm)")
        self.ax.set_ylabel("Y (μm)")
        self.ax.set_title(
            'Click to select fiducial markers\n(Press "q" when finished, "r" to remove last)'
        )

        # Connect event handlers
        self.fig.canvas.mpl_connect("button_press_event", self._on_click)
        self.fig.canvas.mpl_connect("key_press_event", self._on_key_press)

        plt.show()

        # Process selections
        return self._process_selections(pick_radius_nm, min_localizations)

    def _on_click(self, event):
        if event.inaxes != self.ax:
            return

        x, y = event.xdata, event.ydata
        if x is None or y is None:
            return

        self.picked_coordinates.append((x, y))

        # Draw selection circle
        circle = Circle((x, y), 0.5, fill=False, color="cyan", linewidth=2)
        self.ax.add_patch(circle)
        self.circles.append(circle)

        # Add label
        text = self.ax.text(
            x,
            y + 0.6,
            f"{len(self.picked_coordinates)}",
            color="cyan",
            ha="center",
            va="bottom",
            fontweight="bold",
        )
        self.circles.append(text)

        self.fig.canvas.draw()
        print(f"Selected fiducial {len(self.picked_coordinates)} at ({x:.2f}, {y:.2f})")

    def _on_key_press(self, event):
        if event.key == "r" and len(self.picked_coordinates) > 0:
            # Remove last selection
            self.picked_coordinates.pop()

            if len(self.circles) >= 2:
                self.circles[-1].remove()  # text
                self.circles[-2].remove()  # circle
                self.circles = self.circles[:-2]

            self.fig.canvas.draw()
            print(
                f"Removed last selection. {len(self.picked_coordinates)} fiducials remaining"
            )

        elif event.key == "q":
            plt.close(self.fig)
            print("Selection finished")

    def _process_selections(self, pick_radius_nm, min_localizations):
        """Process the manually selected coordinates."""

        if len(self.picked_coordinates) == 0:
            print("❌ No fiducials selected")
            return None

        print(f"\nProcessing {len(self.picked_coordinates)} selected fiducials...")

        pick_radius_pixels = pick_radius_nm / (self.pixelsize * 1000)
        valid_picks = []
        valid_coords = []

        for i, (x, y) in enumerate(self.picked_coordinates):
            # Find localizations within radius
            distances = np.sqrt((self.locs.xc - x) ** 2 + (self.locs.yc - y) ** 2)
            within_radius = self.locs[distances <= pick_radius_pixels]

            if len(within_radius) >= min_localizations:
                valid_picks.append(within_radius)
                valid_coords.append((x, y))
                print(f"✅ Fiducial {i+1}: {len(within_radius)} localizations")
            else:
                print(
                    f"⚠️ Fiducial {i+1}: Only {len(within_radius)} localizations (< {min_localizations})"
                )

        if len(valid_coords) == 0:
            print("❌ No valid fiducials found")
            return None

        # Create localization array with group field
        selected_locs = self._create_grouped_localizations(valid_picks)

        result = FiducialSelectionResult(
            picked_coordinates=valid_coords,
            selected_localizations=selected_locs,
            n_fiducials=len(valid_coords),
            selection_method="manual",
            parameters_used={
                "pick_radius_nm": pick_radius_nm,
                "min_localizations": min_localizations,
            },
        )

        print(f"✅ Manual selection completed: {len(valid_coords)} valid fiducials")
        return result

    def _create_grouped_localizations(self, picked_locs_list):
        """Create localization array with group field."""

        # Start with all localizations, group = -1 (not fiducial)
        group_field = np.full(len(self.locs), -1, dtype=np.int32)

        # Assign group IDs to picked localizations
        for group_id, fiducial_locs in enumerate(picked_locs_list):
            for fid_loc in fiducial_locs:
                # Find matching localizations
                matches = (
                    (self.locs.frame == fid_loc.frame)
                    & (np.abs(self.locs.xc - fid_loc.xc) < 0.01)
                    & (np.abs(self.locs.yc - fid_loc.yc) < 0.01)
                )
                group_field[matches] = group_id

        # Create new recarray with group field
        original_dtype = self.locs.dtype
        group_dtype = np.dtype(original_dtype.descr + [("group", "i4")])

        new_locs = np.empty(len(self.locs), dtype=group_dtype)

        # Copy original data
        for field in original_dtype.names:
            new_locs[field] = self.locs[field]

        # Add group data
        new_locs["group"] = group_field

        return new_locs.view(np.recarray)


# Uncomment the lines below to try manual selection:
#
# manual_selector = ManualFiducialSelector(locs, info)
# manual_result = manual_selector.start_selection(
#     pick_radius_nm=800.0,     # Selection radius
#     min_localizations=50      # Minimum localizations per fiducial
# )
#
# if manual_result is not None:
#     # Perform drift correction with manually selected fiducials
#     drift_corrector = DCF.Drift_Correction_Functions()
#     manual_corrected, manual_drift = drift_corrector.undrift(
#         locs=manual_result.selected_localizations,
#         info=info,
#         method="fiducial"
#     )
#
#     print(f"✅ Manual fiducial drift correction completed!")
#     preview_fiducials(locs, info, manual_result)

print("Manual selection code ready (uncomment to use)")

## Summary and Next Steps

This notebook has demonstrated the complete workflow for fiducial-based drift correction:

### ✅ What We Accomplished:
1. **Data Loading**: Created/loaded localization data with simulated drift
2. **Fiducial Detection**: Automatically identified fiducial markers using optimized parameters
3. **Drift Correction**: Applied fiducial-based drift correction to remove measured drift
4. **Quality Analysis**: Assessed fiducial precision and drift correction effectiveness
5. **Visualization**: Created comprehensive plots showing before/after comparison
6. **Export**: Saved all results for further analysis

### 📊 Key Results:
- **Fiducials detected**: Check the quality metrics above
- **Drift corrected**: Maximum drift magnitude and range shown in plots
- **Precision achieved**: Fiducial localization precision in nanometers

### 🔧 For Your Own Data:
1. Replace the `create_example_data()` function with your own data loading
2. Adjust detection parameters based on your fiducial characteristics:
   - `threshold_percentile`: Lower for more candidates, higher for fewer
   - `box_size_nm`: Larger for bigger fiducials, smaller for tight spots
   - `min_frames_fraction`: Lower if fiducials don't appear in all frames
3. Use manual selection if automatic detection fails

### 🚀 Advanced Usage:
- **Multi-color data**: Apply the same fiducial selection to multiple color channels
- **3D data**: The method supports z-drift correction if z-coordinates are available
- **Time-lapse**: Process long time-lapse datasets by adjusting `min_frames_fraction`
- **Custom analysis**: Use the exported data for custom drift analysis workflows

### 📁 Generated Files:
All results are saved with descriptive filenames for easy identification and further analysis.

In [ ]:
# Final summary
print("\n" + "=" * 60)
print("FIDUCIAL DRIFT CORRECTION WORKFLOW COMPLETE")
print("=" * 60)

print(f"✅ Method used: {auto_result.selection_method}")
print(f"✅ Fiducials detected: {auto_result.n_fiducials}")
print(f"✅ Original localizations: {len(locs):,}")
print(f"✅ Corrected localizations: {len(auto_result.selected_localizations):,}")
print(
    f"✅ Maximum drift corrected: {quality_metrics.get('max_drift_magnitude', 0):.3f} pixels"
)
print(
    f"✅ Mean fiducial precision: {quality_metrics.get('mean_precision_x_nm', 0):.0f} nm (X), {quality_metrics.get('mean_precision_y_nm', 0):.0f} nm (Y)"
)
print(f"✅ Files saved: {len(saved_files)} total")

print("\n🎉 Ready for super-resolution analysis!")
print("\nFor questions or issues, see the DriftCorrectionFunctions.py documentation.")